# CoT · ReAct · Reflexion 논문 재현 노트북 (Ollama 8B, Colab)

세 논문의 방법을 **논문과 공식 코드의 프롬프트 그대로** 로컬 모델에서 돌려 보는 노트북입니다.

| 논문 | 이 노트북에서 재현하는 것 | 출처 |
|---|---|---|
| CoT (Wei et al. 2022) | 수학 문제에서 Standard 대 CoT 프롬프팅 비교. few-shot 예시 8개 | 논문 Table 20 |
| ReAct (Yao et al. 2023) | HotpotQA에서 Standard / CoT / Act / ReAct 비교. `search` `lookup` `finish` 세 행동, 최대 7스텝 | 공식 저장소 `ysymyth/ReAct` |
| Reflexion (Shinn et al. 2023) | ReAct를 Actor로 두고, 정확 일치(EM) 평가 → 자기 성찰 → 메모리(최근 3개) → 재시도 | 공식 저장소 `noahshinn/reflexion` |

### 실행 순서
위에서부터 차례로 실행합니다. **GPU 런타임**으로 두세요 (런타임 → 런타임 유형 변경). 8B 모델은 T4 이상이면 되고, A100이면 더 빠릅니다.

① Ollama·모델 준비 → ② 데이터 파일 → ③ 코어 파일 → ④ 점검 → ⑤ CoT 실험 → ⑥ 한 문제 4가지 방법 비교 → ⑦ Reflexion → ⑧ 묶음 실험·표 → ⑨ 결과 내려받기

### 논문과 다른 점 (이 두 가지뿐입니다)
1. **모델**: 논문의 PaLM-540B·GPT 계열 대신 Ollama 로컬 모델(기본 `llama3.1:8b`)을 씁니다.
2. **Wikipedia 접속 방식**: 논문 코드는 웹 페이지를 긁어 오지만, 여기서는 MediaWiki API를 씁니다. 반환 규칙(문서가 있으면 첫 5문장, 없으면 비슷한 제목 5개, `lookup`은 키워드가 든 다음 문장)은 논문 코드와 같습니다.

### 이 노트북의 원칙
- **실행 오류는 막습니다.** 통신 실패, 형식이 깨진 출력, 시간 초과는 예외로 멈추지 않고 결과의 `status`와 표의 "실행 실패" 칸에 기록됩니다.
- **모델의 오답은 고치지 않습니다.** 근거 검증, 웹 검색 보조, 답 보정 같은 장치는 논문에 없으므로 넣지 않았습니다. 8B 모델이 틀리는 것은 버그가 아니라 실험 결과입니다 (CoT 논문은 추론 능력이 약 100B 규모에서 나타난다고 보고합니다).
- 필요한 패키지는 `requests` 하나뿐이라 버전 충돌이 날 일이 없습니다. LangChain·Streamlit은 쓰지 않습니다.

## ① Ollama 서버와 모델 준비
처음 실행하면 모델(약 4.9GB)을 내려받느라 몇 분 걸립니다. 다시 실행해도 안전합니다.

In [ ]:
import os, shutil, subprocess, time
import requests

MODEL_NAME = "llama3.1:8b"          # 다른 모델을 쓰려면 여기만 바꾸세요 (예: "qwen2.5:7b")
OLLAMA_URL = "http://127.0.0.1:11434"
SETUP_OK = False


def _sh(cmd, timeout=1800):
    """명령을 실행하고 (성공 여부, 출력 끝부분)을 돌려준다. 예외를 던지지 않는다."""
    try:
        p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout)
        return p.returncode == 0, (p.stdout + p.stderr)[-800:]
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


def _server_up():
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).status_code == 200
    except Exception:
        return False


def _models():
    try:
        return [m.get("name", "") for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json().get("models", [])]
    except Exception:
        return []


def _has_model(name):
    return any(m == name or m.startswith(name + ":") or m.split(":")[0] == name for m in _models())


def setup_ollama():
    # 0) GPU 확인 (없어도 돌지만 매우 느립니다)
    ok, out = _sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
    print("GPU:", out.strip() if ok else "없음 → 런타임 유형을 GPU로 바꾸는 것을 권합니다 (CPU는 매우 느립니다).")

    # 1) 설치
    if shutil.which("ollama") is None:
        print("⏳ Ollama 설치 중...")
        _sh("apt-get -qq update && apt-get -qq install -y zstd pciutils curl", timeout=600)
        for attempt in range(3):
            ok, out = _sh("curl -fsSL https://ollama.com/install.sh | sh", timeout=900)
            if shutil.which("ollama"):
                break
            time.sleep(5)
        if shutil.which("ollama") is None:
            print("⛔ Ollama 설치 실패. 네트워크 문제일 수 있습니다. 이 셀을 다시 실행하세요.\n", out)
            return False

    # 2) 서버 (이미 떠 있으면 그대로 사용)
    if not _server_up():
        env = dict(os.environ, OLLAMA_KEEP_ALIVE="30m")
        try:
            subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, env=env)
        except Exception as exc:
            print("⛔ Ollama 서버를 시작하지 못했습니다:", exc)
            return False
        for _ in range(60):
            if _server_up():
                break
            time.sleep(1)
    if not _server_up():
        tail = open("ollama.log").read()[-800:] if os.path.exists("ollama.log") else ""
        print("⛔ Ollama 서버가 응답하지 않습니다. 이 셀을 다시 실행하세요.\n", tail)
        return False
    print("✅ Ollama 서버 정상")

    # 3) 모델
    for attempt in range(3):
        if _has_model(MODEL_NAME):
            break
        print(f"⏳ {MODEL_NAME} 내려받는 중 (시도 {attempt + 1}/3)...")
        _sh(["ollama", "pull", MODEL_NAME], timeout=3600)
    if not _has_model(MODEL_NAME):
        print(f"⛔ 모델 {MODEL_NAME} 을(를) 받지 못했습니다. 현재 모델: {_models()}. 이 셀을 다시 실행하세요.")
        return False

    # 4) 워밍업 (실험과 같은 컨텍스트 크기로 미리 올려 둠)
    t0 = time.time()
    try:
        requests.post(f"{OLLAMA_URL}/api/generate", timeout=600, json={
            "model": MODEL_NAME, "prompt": "hi", "stream": False, "keep_alive": "30m",
            "options": {"num_ctx": 8192, "num_predict": 1}})
    except Exception as exc:
        print("⚠️ 워밍업 호출 실패 (계속 진행합니다):", type(exc).__name__)
    print(f"✅ 모델 준비 완료: {MODEL_NAME} (워밍업 {time.time() - t0:.0f}초)")
    return True


SETUP_OK = setup_ollama()

## ② 데이터 파일 (`paper_data.py`)
논문과 공식 저장소에서 가져온 프롬프트와 문제입니다. 내용을 바꾸면 논문 재현이 아니게 되므로 그대로 두세요.

In [ ]:
%%writefile paper_data.py
"""paper_data.py — 논문·공식 저장소에서 그대로 가져온 프롬프트와 문제 (수정하지 마세요)

출처
  - REACT_* / ACT_* / HOTPOT_*_EXAMPLES : github.com/ysymyth/ReAct (MIT) prompts/prompts_naive.json, hotpotqa.ipynb
  - REFLECT_* / REFLECTION_HEADER       : github.com/noahshinn/reflexion (MIT) hotpotqa_runs/prompts.py, fewshots.py
  - MATH_COT_EXAMPLES                   : Wei et al. 2022, Table 20 (수학 문제용 예시 8개)
    MATH_STANDARD_EXAMPLES              : 같은 8문제에서 풀이 과정만 뺀 것 (논문의 standard prompting 비교군)
  - HOTPOT_DEV  : HotpotQA dev (CC BY-SA 4.0). ReAct 논문 코드와 같은 순서(random.Random(233) 셔플)의 앞 30문제
  - GSM8K_TEST  : GSM8K test (MIT) 앞 20문제
원본의 깨진 문자(en dash) 두 군데만 바로잡았고, 나머지는 원문 그대로입니다.
"""
REACT_INSTRUCTION = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types: 
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""

REACT_EXAMPLES = """
Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought 1: I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.
Action 1: Search[Colorado orogeny]
Observation 1: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.
Thought 2: It does not mention the eastern sector. So I need to look up eastern sector.
Action 2: Lookup[eastern sector]
Observation 2: (Result 1 / 1) The eastern sector extends into the High Plains and is called the Central Plains orogeny.
Thought 3: The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.
Action 3: Search[High Plains]
Observation 3: High Plains refers to one of two distinct land regions:
Thought 4: I need to instead search High Plains (United States).
Action 4: Search[High Plains (United States)]
Observation 4: The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130 m).[3]
Thought 5: High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.
Action 5: Finish[1,800 to 7,000 ft]
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Thought 1: The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.
Action 1: Search[Milhouse]
Observation 1: Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.
Thought 2: The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".
Action 2: Lookup[named after]
Observation 2: (Result 1 / 1) Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous. 
Thought 3: Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.
Action 3: Finish[Richard Nixon]
Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Thought 1: I need to search Adam Clayton Powell and The Saimaa Gesture, and find which documentary is about Finnish rock groups.
Action 1: Search[Adam Clayton Powell]
Observation 1: Could not find [Adam Clayton Powell]. Similar: ['Adam Clayton Powell III', 'Seventh Avenue (Manhattan)', 'Adam Clayton Powell Jr. State Office Building', 'Isabel Washington Powell', 'Adam Powell', 'Adam Clayton Powell (film)', 'Giancarlo Esposito'].
Thought 2: To find the documentary, I can search Adam Clayton Powell (film).
Action 2: Search[Adam Clayton Powell (film)]
Observation 2: Adam Clayton Powell is a 1989 American documentary film directed by Richard Kilberg.
The film is about the rise and fall of influential African-American politician Adam Clayton Powell Jr.[3][4] It was later aired as part of the PBS series The American Experience.
Thought 3: Adam Clayton Powell (film) is a documentary about an African-American politician, not Finnish rock groups. So the documentary about Finnish rock groups must instead be The Saimaa Gesture.
Action 3: Finish[The Saimaa Gesture]
Question: What profession does Nicholas Ray and Elia Kazan have in common?
Thought 1: I need to search Nicholas Ray and Elia Kazan, find their professions, then find the profession they have in common.
Action 1: Search[Nicholas Ray]
Observation 1: Nicholas Ray (born Raymond Nicholas Kienzle Jr., August 7, 1911 – June 16, 1979) was an American film director, screenwriter, and actor best known for the 1955 film Rebel Without a Cause.
Thought 2: Professions of Nicholas Ray are director, screenwriter, and actor. I need to search Elia Kazan next and find his professions.
Action 2: Search[Elia Kazan]
Observation 2: Elia Kazan was an American film and theatre director, producer, screenwriter and actor.
Thought 3: Professions of Elia Kazan are director, producer, screenwriter, and actor. So profession Nicholas Ray and Elia Kazan have in common is director, screenwriter, and actor.
Action 3: Finish[director, screenwriter, actor]
Question: Which magazine was started first Arthur's Magazine or First for Women?
Thought 1: I need to search Arthur's Magazine and First for Women, and find which was started first.
Action 1: Search[Arthur's Magazine]
Observation 1: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. 
Thought 2: Arthur's Magazine was started in 1844. I need to search First for Women next.
Action 2: Search[First for Women]
Observation 2: First for Women is a woman's magazine published by Bauer Media Group in the USA.[1] The magazine was started in 1989. 
Thought 3: First for Women was started in 1989. 1844 (Arthur's Magazine) < 1989 (First for Women), so Arthur's Magazine was started first.
Action 3: Finish[Arthur's Magazine]
Question: Were Pavel Urysohn and Leonid Levin known for the same type of work?
Thought 1: I need to search Pavel Urysohn and Leonid Levin, find their types of work, then find if they are the same.
Action 1: Search[Pavel Urysohn]
Observation 1: Pavel Samuilovich Urysohn (February 3, 1898 – August 17, 1924) was a Soviet mathematician who is best known for his contributions in dimension theory.
Thought 2: Pavel Urysohn is a mathematician. I need to search Leonid Levin next and find its type of work.
Action 2: Search[Leonid Levin]
Observation 2: Leonid Anatolievich Levin is a Soviet-American mathematician and computer scientist. 
Thought 3: Leonid Levin is a mathematician and computer scientist. So Pavel Urysohn and Leonid Levin have the same type of work. 
Action 3: Finish[yes]
"""

ACT_EXAMPLES = """Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Action 1: Search[Colorado orogeny]
Observation 1: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.
Action 2: Lookup[eastern sector]
Observation 2: (Result 1 / 1) The eastern sector extends into the High Plains and is called the Central Plains orogeny.
Action 3: Search[High Plains]
Observation 3: High Plains refers to one of two distinct land regions:
Action 4: Search[High Plains (United States)]
Observation 4: The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130 m).[3]
Action 5: Finish[1,800 to 7,000 ft]
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Action 1: Search[Milhouse]
Observation 1: Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.
Action 2: Lookup[named after]
Observation 2: (Result 1 / 1) Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous. 
Action 3: Finish[Richard Nixon]
Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Action 1: Search[Adam Clayton Powell]
Observation 1: Could not find [Adam Clayton Powell]. Similar: ['Adam Clayton Powell III', 'Seventh Avenue (Manhattan)', 'Adam Clayton Powell Jr. State Office Building', 'Isabel Washington Powell', 'Adam Powell', 'Adam Clayton Powell (film)', 'Giancarlo Esposito'].
Action 2: Search[Adam Clayton Powell (film)]
Observation 2: Adam Clayton Powell is a 1989 American documentary film directed by Richard Kilberg.
The film is about the rise and fall of influential African-American politician Adam Clayton Powell Jr.[3][4] It was later aired as part of the PBS series The American Experience.
Action 3: Finish[The Saimaa Gesture]
Question: What profession does Nicholas Ray and Elia Kazan have in common?
Action 1: Search[Nicholas Ray]
Observation 1: Nicholas Ray (born Raymond Nicholas Kienzle Jr., August 7, 1911 – June 16, 1979) was an American film director, screenwriter, and actor best known for the 1955 film Rebel Without a Cause.
Action 2: Search[Elia Kazan]
Observation 2: Elia Kazan was an American film and theatre director, producer, screenwriter and actor.
Action 3: Finish[director, screenwriter, actor]
Question: Which magazine was started first Arthur's Magazine or First for Women?
Action 1: Search[Arthur's Magazine]
Observation 1: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. 
Action 2: Search[First for Women]
Observation 2: First for Women is a woman's magazine published by Bauer Media Group in the USA.[1] The magazine was started in 1989. 
Action 3: Finish[Arthur's Magazine]
Question: Were Pavel Urysohn and Leonid Levin known for the same type of work?
Action 1: Search[Pavel Urysohn]
Observation 1: Pavel Samuilovich Urysohn (February 3, 1898 – August 17, 1924) was a Soviet mathematician who is best known for his contributions in dimension theory.
Action 2: Search[Leonid Levin]
Observation 2: Leonid Anatolievich Levin is a Soviet-American mathematician and computer scientist. 
Action 3: Finish[yes]

"""

HOTPOT_COT_EXAMPLES = """Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought: Let's think step by step. The eastern sector of Colorado orogeny extends into the High Plains. High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.
Answer: 1,800 to 7,000 ft
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Thought: Let's think step by step. Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.
Answer: Richard Nixon
Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Thought: Let's think step by step. Adam Clayton Powell (film) is a documentary about an African-American politician, not Finnish rock groups. So the documentary about Finnish rock groups must instead be The Saimaa Gesture.
Answer: The Saimaa Gesture
Question: What profession does Nicholas Ray and Elia Kazan have in common?
Thought: Let's think step by step. Professions of Nicholas Ray are director, screenwriter, and actor. Professions of Elia Kazan are director, producer, screenwriter, and actor. So profession Nicholas Ray and Elia Kazan have in common is director, screenwriter, and actor.
Answer: director, screenwriter, actor
Question: Which magazine was started first Arthur's Magazine or First for Women?
Thought: Let's think step by step. Arthur's Magazine was started in 1844. First for Women was started in 1989. 1844 (Arthur's Magazine) < 1989 (First for Women), so Arthur's Magazine was started first.
Answer: Arthur's Magazine
Question: Were Pavel Urysohn and Leonid Levin known for the same type of work?
Thought: Let's think step by step. Pavel Urysohn is a mathematician. Leonid Levin is a mathematician and computer scientist. So Pavel Urysohn and Leonid Levin have the same type of work.
Answer: Yes
"""

HOTPOT_STANDARD_EXAMPLES = """Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Answer: 1,800 to 7,000 ft
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Answer: Richard Nixon
Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Answer: The Saimaa Gesture
Question: What profession does Nicholas Ray and Elia Kazan have in common?
Answer: director, screenwriter, actor
Question: Which magazine was started first Arthur's Magazine or First for Women?
Answer: Arthur's Magazine
Question: Were Pavel Urysohn and Leonid Levin known for the same type of work?
Answer: Yes
"""

REFLECTION_HEADER = """You have attempted to answer following question before and failed. The following reflection(s) give a plan to avoid failing to answer the question in the same way you did previously. Use them to improve your strategy of correctly answering the given question.
"""

REFLECT_INSTRUCTION = """You are an advanced reasoning agent that can improve based on self refection. You will be given a previous reasoning trial in which you were given access to an Docstore API environment and a question to answer. You were unsuccessful in answering the question either because you guessed the wrong answer with Finish[<answer>], or you used up your set number of reasoning steps. In a few sentences, Diagnose a possible reason for failure and devise a new, concise, high level plan that aims to mitigate the same failure. Use complete sentences.  
Here are some examples:
{examples}

Previous trial:
Question: {question}{scratchpad}

Reflection:"""

REFLECT_EXAMPLES = """
Previous Trial:
Question: The Rome Protocols were signed by three Prime Ministers one of which was assassinated as part of what?
Thought 1: I need to search Rome Protocols, find the three Prime Ministers, then find what they were assassinated as part of.
Action 1: Search[Rome Protocols]
Observation 1: The Rome Protocols were a series of three international agreements concluded in Rome on 17 March 1934 between the governments of Austria, Hungary and Italy. They were signed by Italian Prime Minister Benito Mussolini, Austrian Chancellor Engelbert Dollfuss and Hungarian Prime Minister Gyula Gömbös. All of the protocols went into effect on 12 July 1934 and were registered in League of Nations Treaty Series on 12 December 1934.
Thought 2: The three Prime Ministers were Benito Mussolini, Engelbert Dollfuss, and Gyula Gömbös. I need to search Benito Mussolini and find what he was assassinated as part of.
Action 2: Search[Benito Mussolini]
Observation 2: Benito Amilcare Andrea Mussolini (UK:  MU(U)SS-ə-LEE-nee, US:  MOOSS-, Italian: [beˈniːto aˈmilkare anˈdrɛːa mussoˈliːni]; 29 July 1883 – 28 April 1945) was an Italian politician and journalist who founded and led the National Fascist Party (PNF). He was Prime Minister of Italy from the March on Rome in 1922 until his deposition in 1943, as well as "Duce" of Italian fascism from the establishment of the Italian Fasces of Combat in 1919 until his summary execution in 1945 by Italian partisans. As dictator of Italy and principal founder of fascism, Mussolini inspired and supported the international spread of fascist movements during the inter-war period.Mussolini was originally a socialist politician and a journalist at the Avanti! newspaper. In 1912, he became a member of the National Directorate of the Italian Socialist Party (PSI), but he was expelled from the PSI for advocating military intervention in World War I, in opposition to the party's stance on neutrality. In 1914, Mussolini founded a new journal, Il Popolo d'Italia, and served in the Royal Italian Army during the war until he was wounded and discharged in 1917. Mussolini denounced the PSI, his views now centering on Italian nationalism instead of socialism, and later founded the fascist movement which came to oppose egalitarianism and class conflict, instead advocating "revolutionary nationalism" transcending class lines. On 31 October 1922, following the March on Rome (28–30 October), Mussolini was appointed prime minister by King Victor Emmanuel III, becoming the youngest individual to hold the office up to that time. After removing all political opposition through his secret police and outlawing labor strikes, Mussolini and his followers consolidated power through a series of laws that transformed the nation into a one-party dictatorship. Within five years, Mussolini had established dictatorial authority by both legal and illegal means and aspired to create a totalitarian state. In 1929, Mussolini signed the Lateran Treaty with the Holy See to establish Vatican City.
Mussolini's foreign policy aimed to restore the ancient grandeur of the Roman Empire by expanding Italian colonial possessions and the fascist sphere of influence. In the 1920s, he ordered the Pacification of Libya, instructed the bombing of Corfu over an incident with Greece, established a protectorate over Albania, and incorporated the city of Fiume into the Italian state via agreements with Yugoslavia. In 1936, Ethiopia was conquered following the Second Italo-Ethiopian War and merged into Italian East Africa (AOI) with Eritrea and Somalia. In 1939, Italian forces annexed Albania. Between 1936 and 1939, Mussolini ordered the successful Italian military intervention in Spain in favor of Francisco Franco during the Spanish Civil War. Mussolini's Italy initially tried to avoid the outbreak of a second global war, sending troops at the Brenner Pass to delay Anschluss and taking part in the Stresa Front, the Lytton Report, the Treaty of Lausanne, the Four-Power Pact and the Munich Agreement. However, Italy then alienated itself from Britain and France by aligning with Germany and Japan. Germany invaded Poland on 1 September 1939, resulting in declarations of war by France and the UK and the start of World War II.
On 10 June 1940, Mussolini decided to enter the war on the Axis side. Despite initial success, the subsequent Axis collapse on multiple fronts and eventual Allied invasion of Sicily made Mussolini lose the support of the population and members of the Fascist Party. As a consequence, early on 25 July 1943, the Grand Council of Fascism passed a motion of no confidence in Mussolini; later that day King Victor Emmanuel III dismissed him as head of government and had him placed in custody, appointing Pietro Badoglio to succeed him as Prime Minister. After the king agreed to an armistice with the Allies, on 12 September 1943 Mussolini was rescued from captivity in the Gran Sasso raid by German paratroopers and Waffen-SS commandos led by Major Otto-Harald Mors. Adolf Hitler, after meeting with the rescued former dictator, then put Mussolini in charge of a puppet regime in northern Italy, the Italian Social Republic (Italian: Repubblica Sociale Italiana, RSI), informally known as the Salò Republic, causing a civil war. In late April 1945, in the wake of near total defeat, Mussolini and his mistress Clara Petacci attempted to flee to Switzerland, but both were captured by Italian communist partisans and summarily executed by firing squad on 28 April 1945 near Lake Como. The bodies of Mussolini and his mistress were then taken to Milan, where they were hung upside down at a service station to publicly confirm their demise.
Thought 3: Benito Mussolini was assassinated as part of the collapse of the Axis powers during World War II. So the Rome Protocols were signed by three Prime Ministers one of which was assassinated as part of World War II.
Action 3: Finish[World War II]

Reflection: I searched one of the prime ministers involved in the signing, then attemted to answer right away. I should have searched each of the prime ministers, then looked up 'death' on each of their pages in order to get more information before answering.

Previous Trial:
Question: Kam Heskin plays Paige Morgan in a 2004 film directed by who?
Thought 1: I need to search Kam Heskin and Paige Morgan, find the 2004 film they are in, then find the director of the film.
Action 1: Search[Kam Heskin]
Observation 1: Kam Erika Heskin (born May 8, 1973) is an American actress. She began her career playing Caitlin Deschanel on the NBC daytime soap opera Sunset Beach (1998–1999), before appearing in films Planet of the Apes (2001) and Catch Me If You Can (2002). Heskin went on in 2003 to play Elizabeth Bennet in an independent film Pride & Prejudice: A Latter-Day Comedy, Robin in Charmed (2004),  and Paige Morgan in The Prince and Me film franchise (2006–2010).
Thought 2: Kam Heskin plays Paige Morgan in The Prince and Me film franchise. I need to search The Prince and Me film franchise and find the director of the 2004 film.
Action 2: Search[The Prince and Me film franchise]
Observation 2: Could not find [The Prince and Me film franchise]. Similar: ['Kam Heskin', 'Prince of Persia', 'Harry Potter and the Half-Blood Prince (film)', 'Prince of Persia: The Sands of Time (film)', 'The Chronicles of Narnia (film series)', 'Majestic Prince (franchise)', 'Kingsman (franchise)', 'The Little Prince (2015 film)', 'Shrek (franchise)', 'Harry Potter (film series)']
Thought 3: I need to search The Prince and Me (2004 film) instead.
Action 3: Search[The Prince and Me (2004 film)]
Observation 3: Could not find [The Prince and Me (2004 film)]. Similar: ['The Prince & Me', 'The Prince & Me 2: The Royal Wedding', 'Kam Heskin', 'Prince of Darkness (film)', 'Prince of Persia: The Sands of Time (film)', 'Rob Knox', 'Alexander (2004 film)', 'Prince of Persia', 'Brooklynn Prince', 'Prince (musician)']
Thought 4: I need to search The Prince & Me (2004 film).
Action 4: Search[The Prince & Me (2004 film)]
Observation 4: Could not find [The Prince & Me (2004 film)]. Similar: ['The Prince & Me', 'The Prince & Me 2: The Royal Wedding', 'Prince of Darkness (film)', 'Prince of Persia: The Sands of Time (film)', 'Rob Knox', 'Alexander (2004 film)', 'Prince (musician)', 'Prince of Persia', 'Kam Heskin', 'Brooklynn Prince']
Thought 5: I need to search The Prince & Me (2004 film) instead.
Action 5: Search[The Prince & Me (2004 film)]
Observation 5: Could not find [The Prince & Me (2004 film)]. Similar: ['The Prince & Me', 'The Prince & Me 2: The Royal Wedding', 'Prince of Darkness (film)', 'Prince of Persia: The Sands of Time (film)', 'Rob Knox', 'Alexander (2004 film)', 'Prince (musician)', 'Prince of Persia', 'Kam Heskin', 'Brooklynn Prince']
Thought 6: I need to search The Prince & Me (2004 film).
Action 6: Search[The Prince & Me (2004 film)]
Observation 6: Could not find [The Prince & Me (2004 film)]. Similar: ['The Prince & Me', 'The Prince & Me 2: The Royal Wedding', 'Prince of Darkness (film)', 'Prince of Persia: The Sands of Time (film)', 'Rob Knox', 'Alexander (2004 film)', 'Prince (musician)', 'Prince of Persia', 'Kam Heskin', 'Brooklynn Prince']

Reflection: I got stuck in a loop where I kept trying to search 'The Prince & Me (2004 film)' but the page could not be found. Instead I should have tried to search the similar results that had a similar name to see and they were made in 2004.
"""

MATH_COT_EXAMPLES = """Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Q: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
A: Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The answer is 39.

Q: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?
A: Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The answer is 8.

Q: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?
A: Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The answer is 9.

Q: There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?
A: There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The answer is 29.

Q: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?
A: Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The answer is 33.

Q: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?
A: Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The answer is 8.

"""

MATH_STANDARD_EXAMPLES = """Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: The answer is 5.

Q: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
A: The answer is 39.

Q: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?
A: The answer is 8.

Q: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?
A: The answer is 9.

Q: There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?
A: The answer is 29.

Q: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?
A: The answer is 33.

Q: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?
A: The answer is 8.

"""

HOTPOT_DEV = [{'idx': 3687,
  'question': 'What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?',
  'answer': 'Beyond the Clouds'},
 {'idx': 6238, 'question': 'Who created the show with Wendy Schaal doing the voice of Francine?', 'answer': 'Seth MacFarlane'},
 {'idx': 5388,
  'question': 'What objects were carried into battle by these naval ships for qhich the QF 6-pounder Hotchkiss were introduced to defend against?',
  'answer': 'torpedoes'},
 {'idx': 3522, 'question': 'Who is older Danny Green or James Worthy?', 'answer': 'James Worthy'},
 {'idx': 3824,
  'question': 'Author David Chanoff has collaborated with a U.S. Navy admiral who served as the ambassador to the United Kingdom under which President?',
  'answer': 'Bill Clinton'},
 {'idx': 2866, 'question': 'What was the first year a scientific journal published by an organization located in the Bronx was published?', 'answer': '1909'},
 {'idx': 1551, 'question': 'Geoff LaTulippe is an American writer whose best-known work was directed by whom?', 'answer': 'Nanette Burstein'},
 {'idx': 2544, 'question': 'Are Hoodoo Gurus and Pierre Bouvier of the same nationality?', 'answer': 'no'},
 {'idx': 1557,
  'question': 'Hardley Flood is an area of lagoons that support populations of the waterfowl that are a medium sized version of what animal?',
  'answer': 'duck'},
 {'idx': 5762, 'question': 'The Twelfth United States Army Group commander was the first chairman of what?', 'answer': 'Joint Chiefs of Staff'},
 {'idx': 7275, 'question': 'Which actor and film director is from German Jewish descent, Kurt Gerron or Dell Henderson?', 'answer': 'Kurt Gerron'},
 {'idx': 3188, 'question': 'Michael Braz completed a libretto for what Georgia college, founded in 1906?', 'answer': 'A Scholar Under Siege'},
 {'idx': 200, 'question': 'The 2000 British film Snatch was later adapted into a television series for what streaming service?', 'answer': 'Crackle'},
 {'idx': 2825,
  'question': 'Kurt Fuller from Stockton, CA appeared in which American Detective Comedy Drama television series created by Steve Franks?',
  'answer': 'Psych is an American detective comedy-drama'},
 {'idx': 3543,
  'question': "The central figure used in the corporate branding of Singapore Airlines has been an element of it's marketing since what year?",
  'answer': '1972'},
 {'idx': 1320, 'question': 'Who was older, Andrew Preston or James Taylor?', 'answer': 'James Taylor'},
 {'idx': 2811, 'question': 'How many households does the network that Di-Gata Defenders plays on in South Africa reach?', 'answer': '1.5 million households'},
 {'idx': 1885, 'question': 'What star of Parks and Recreation appeared in November?', 'answer': 'Nick Offerman'},
 {'idx': 4237, 'question': '"Tunak", is a bhangra/pop love song by an artist born in which year ?', 'answer': '1967'},
 {'idx': 7026, 'question': 'Which peak is flanked by Manaslu, Shispare or Ngadi Chuli?', 'answer': 'Ngadi Chuli'},
 {'idx': 1720,
  'question': 'What was the middle name of the actress who was cast alongside Harvey Fierstein, Linda Hart, Dick Latessa, Clarke Thorell, Mary Bond Davis, '
              'Laura Bell Bundy, Matthew Morrison, Corey Reynolds, and Marissa Jaret Winokur in Hairspray?',
  'answer': 'Marie'},
 {'idx': 1560, 'question': 'According to the 2011 census what is the population of the city where Royal Wahingdoh F.C. are based ?', 'answer': '143,007'},
 {'idx': 4525,
  'question': 'The Ranters were a loose collection of radical so-called heretics whose main objection was with what major division of Christendom?',
  'answer': 'Roman Catholicism'},
 {'idx': 2609,
  'question': 'Ruth Vollmer begins work designing window displays for which American luxury jewelry and specialty retailer, headquartered in New York City',
  'answer': "Tiffany's"},
 {'idx': 4506, 'question': 'What city does the company that owns Bumble and bumble have its headquarters in?', 'answer': 'New York City'},
 {'idx': 3395, 'question': 'What languages did the son of Sacagawea speak?', 'answer': 'French and English'},
 {'idx': 5123, 'question': 'Ferdinand Zvonimir von Habsburg is the eldest son of the head of which House', 'answer': 'House of Habsburg-Lorraine'},
 {'idx': 5503,
  'question': 'What schools of thought are drawn from in the academic discipline of Hafiz Rashid Khan?',
  'answer': 'postmodern schools of thought'},
 {'idx': 807, 'question': 'Alfie Allen played Theon Greyjoy on which show?', 'answer': 'Game of Thrones'},
 {'idx': 3511, 'question': 'Was Vanderbilt University or Emory University founded first?', 'answer': 'Emory University.'}]

GSM8K_TEST = [{'question': 'Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells '
              "the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'answer': 18.0},
 {'question': 'A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?', 'answer': 3.0},
 {'question': 'Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house '
              'by 150%.  How much profit did he make?',
  'answer': 70000.0},
 {'question': 'James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?', 'answer': 540.0},
 {'question': 'Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them '
              'healthy.  She gives the chickens their feed in three separate meals. In the morning, she gives her flock of chickens 15 cups of feed.  In the '
              'afternoon, she gives her chickens another 25 cups of feed.  How many cups of feed does she need to give her chickens in the final meal of the '
              "day if the size of Wendi's flock is 20 chickens?",
  'answer': 20.0},
 {'question': 'Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar '
              'wants to buy 16 glasses. How much does he need to pay for them?',
  'answer': 64.0},
 {'question': 'Toulouse has twice as many sheep as Charleston. Charleston has 4 times as many sheep as Seattle. How many sheep do Toulouse, Charleston, and '
              'Seattle have together if Seattle has 20 sheep?',
  'answer': 260.0},
 {'question': 'Carla is downloading a 200 GB file. Normally she can download 2 GB/minute, but 40% of the way through the download, Windows forces a restart to '
              'install updates, which takes 20 minutes. Then Carla has to restart the download from the beginning. How load does it take to download the file?',
  'answer': 160.0},
 {'question': 'John drives for 3 hours at a speed of 60 mph and then turns around because he realizes he forgot something very important at home.  He tries to '
              'get home in 4 hours but spends the first 2 hours in standstill traffic.  He spends the next half-hour driving at a speed of 30mph, before being '
              'able to drive the remaining time of the 4 hours going at 80 mph.  How far is he from home at the end of those 4 hours?',
  'answer': 45.0},
 {'question': "Eliza's rate per hour for the first 40 hours she works each week is $10. She also receives an overtime pay of 1.2 times her regular hourly "
              'rate. If Eliza worked for 45 hours this week, how much are her earnings for this week?',
  'answer': 460.0},
 {'question': 'A new program had 60 downloads in the first month. The number of downloads in the second month was three times as many as the downloads in the '
              'first month, but then reduced by 30% in the third month. How many downloads did the program have total over the three months?',
  'answer': 366.0},
 {'question': 'Toula went to the bakery and bought various types of pastries. She bought 3 dozen donuts which cost $68 per dozen, 2 dozen mini cupcakes which '
              'cost $80 per dozen, and 6 dozen mini cheesecakes for $55 per dozen. How much was the total cost?',
  'answer': 694.0},
 {'question': 'Carlos is planting a lemon tree. The tree will cost $90 to plant. Each year it will grow 7 lemons, which he can sell for $1.5 each. It costs $3 '
              'a year to water and feed the tree. How many years will it take before he starts earning money on the lemon tree?',
  'answer': 13.0},
 {'question': 'Melanie is a door-to-door saleswoman. She sold a third of her vacuum cleaners at the green house, 2 more to the red house, and half of what was '
              'left at the orange house. If Melanie has 5 vacuum cleaners left, how many did she start with?',
  'answer': 18.0},
 {'question': 'In a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in '
              'hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?',
  'answer': 60.0},
 {'question': 'A merchant wants to make a choice of purchase between 2 purchase plans: jewelry worth $5,000 or electronic gadgets worth $8,000. His financial '
              'advisor speculates that the jewelry market will go up 2.5% while the electronic gadgets market will rise 1.2% within the same month. If the '
              'merchant is looking to maximize profit at the end of this month by making a choice, how much profit would this be?',
  'answer': 125.0},
 {'question': 'Two trains leave San Rafael at the same time. They begin traveling westward, both traveling for 80 miles. The next day, they travel northwards, '
              "covering 150 miles. What's the distance covered by each train in the two days?",
  'answer': 230.0},
 {'question': 'Jill gets paid $20 per hour to teach and $30 to be a cheerleading coach. If she works 50 weeks a year, 35 hours a week as a teacher and 15 '
              "hours a week as a coach, what's her annual salary?",
  'answer': 57500.0},
 {'question': 'Claire makes a 3 egg omelet every morning for breakfast.  How many dozens of eggs will she eat in 4 weeks?', 'answer': 7.0},
 {'question': 'Marissa is hiking a 12-mile trail. She took 1 hour to walk the first 4 miles, then another hour to walk the next two miles. If she wants her '
              'average speed to be 4 miles per hour, what speed (in miles per hour) does she need to walk the remaining distance?',
  'answer': 6.0}]

## ③ 코어 파일 (`paper_core.py`)
세 논문의 루프, Wikipedia 환경, 평가자, 자가 점검이 들어 있습니다.

In [ ]:
%%writefile paper_core.py
"""paper_core.py — CoT / ReAct / Reflexion 논문 재현 코어 (의존성: requests 뿐)

논문과 공식 코드에서 그대로 가져온 것
  - CoT  (Wei et al. 2022)      : 수학 문제용 few-shot 예시 8개 (논문 Table 20), "The answer is N." 형식
  - ReAct (Yao et al. 2023)     : 지시문, 예시 6개, Thought/Action/Observation 루프(최대 7스텝),
                                  search / lookup / finish 환경 규칙  (github.com/ysymyth/ReAct)
  - Reflexion (Shinn et al. 2023): 자기 성찰 프롬프트·예시, 정확 일치(EM) 평가자, 반성 메모리
                                  (github.com/noahshinn/reflexion, hotpotqa_runs)

논문과 다른 것 (환경 제약 때문에 바꾼 것만)
  - 모델: PaLM-540B / GPT 대신 Ollama 로컬 모델
  - Wikipedia: HTML 스크래핑 대신 MediaWiki API (같은 규칙을 재현, 차단·시간초과에 강함)

설계 원칙: 실패는 예외로 터뜨리지 않고 결과의 status 로 기록한다.
           모델이 틀린 답을 내는 것은 고치지 않는다 (그것이 실험 결과다).
"""
from __future__ import annotations

import json
import os
import re
import string
import time
import traceback

import requests

import paper_data as D

# ---------------------------------------------------------------------------
# 0. 설정
# ---------------------------------------------------------------------------
OLLAMA_URL = "http://127.0.0.1:11434"

CFG = {
    "model": "llama3.1:8b",
    # "completion": 템플릿 없이 프롬프트를 그대로 이어 쓰게 함 (논문이 쓴 completion API와 같은 방식)
    # "chat"      : 모델의 채팅 템플릿에 프롬프트를 사용자 메시지로 넣음
    "prompt_mode": "completion",
    "num_ctx": 8192,
    "temperature": 0.0,      # 논문 코드: temperature=0
    "seed": 0,
    "timeout": 300,          # LLM 호출 1회 제한(초)
    "llm_retries": 3,
}

STATS = {"llm_calls": 0, "llm_failures": 0, "wiki_calls": 0, "wiki_failures": 0}


class LLMError(RuntimeError):
    pass


class WikiError(RuntimeError):
    pass


# ---------------------------------------------------------------------------
# 1. LLM (Ollama REST API 직접 호출)
# ---------------------------------------------------------------------------
def ollama_llm(prompt: str, stop=("\n",), max_tokens: int = 100) -> str:
    """프롬프트 뒤에 이어질 텍스트를 반환. 실패하면 LLMError (상위 루프가 잡아서 status 로 기록)."""
    payload = {
        "model": CFG["model"],
        "prompt": prompt,
        "stream": False,
        "raw": CFG["prompt_mode"] == "completion",
        "keep_alive": "30m",
        "options": {
            "temperature": CFG["temperature"],
            "top_p": 1,
            "seed": CFG["seed"],
            "num_predict": int(max_tokens),
            "num_ctx": int(CFG["num_ctx"]),
            "stop": [s for s in (stop or []) if s],
        },
    }
    last = ""
    for attempt in range(int(CFG["llm_retries"])):
        STATS["llm_calls"] += 1
        try:
            r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=CFG["timeout"])
            if r.status_code == 200:
                data = r.json()
                if isinstance(data, dict) and "response" in data:
                    return str(data.get("response") or "")
                last = f"unexpected body: {str(data)[:120]}"
            else:
                last = f"HTTP {r.status_code}: {r.text[:120]}"
        except Exception as exc:  # 네트워크, 시간초과, JSON 오류 등
            last = f"{type(exc).__name__}: {exc}"
        STATS["llm_failures"] += 1
        time.sleep(2 * (attempt + 1))
    raise LLMError(last)


def est_tokens(text: str) -> int:
    """토크나이저 없이 쓰는 보수적 토큰 수 추정 (영문 평균 약 4자/토큰 → 3자로 넉넉히 잡음)."""
    return len(text) // 3 + 1


def _fits(prompt: str, reserve: int = 300) -> bool:
    return est_tokens(prompt) + reserve <= int(CFG["num_ctx"])


def _strip_label(text: str, label: str) -> str:
    """모델이 'Thought 1:' 같은 라벨을 한 번 더 써서 시작하면 그 라벨만 떼어낸다 (내용은 건드리지 않음)."""
    t = text.strip()
    if t.lower().startswith(label.lower()):
        t = t[len(label):].strip()
    return t


# ---------------------------------------------------------------------------
# 2. 평가: 정확 일치 (ReAct·Reflexion 공식 코드와 동일)
# ---------------------------------------------------------------------------
def normalize_answer(s) -> str:
    s = "" if s is None else str(s)
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def EM(answer, key) -> bool:
    return normalize_answer(answer) == normalize_answer(key)


# ---------------------------------------------------------------------------
# 3. Wikipedia 환경 (ReAct wikienv.py 의 규칙을 MediaWiki API 로 재현)
# ---------------------------------------------------------------------------
_WIKI = requests.Session()
_WIKI.headers.update({
    "User-Agent": "PaperReproNotebook/1.0 (educational reproduction of ReAct/Reflexion; python-requests)",
    "Accept": "application/json",
})
_WIKI_API = "https://en.wikipedia.org/w/api.php"
_PAGE_CACHE: dict = {}


def _wiki_get(**params) -> dict:
    params = {"format": "json", "formatversion": 2, **params}
    last = ""
    for attempt in range(4):
        STATS["wiki_calls"] += 1
        try:
            r = _WIKI.get(_WIKI_API, params=params, timeout=10)
            if r.status_code == 200:
                data = r.json()
                if isinstance(data, dict) and "error" not in data:
                    return data
                last = f"api error: {str(data.get('error'))[:120] if isinstance(data, dict) else 'non-dict'}"
                if isinstance(data, dict):      # 요청 자체가 잘못된 경우는 재시도해도 같다
                    break
            else:
                last = f"HTTP {r.status_code}"
        except Exception as exc:
            last = f"{type(exc).__name__}"
        time.sleep(1.5 * (attempt + 1))
    STATS["wiki_failures"] += 1
    raise WikiError(last)


def _search_titles(query: str, limit: int = 5) -> list:
    data = _wiki_get(action="query", list="search", srsearch=query, srlimit=limit, srprop="")
    hits = (data.get("query") or {}).get("search") or []
    return [str(h.get("title", "")) for h in hits if h.get("title")]


def _load_page(title: str):
    """정확한 제목의 문서 본문을 반환. 없거나 동음이의 문서면 None."""
    data = _wiki_get(action="query", prop="extracts|pageprops", explaintext=1, exsectionformat="wiki",
                     redirects=1, ppprop="disambiguation", titles=title)
    pages = (data.get("query") or {}).get("pages") or []
    if not pages:
        return None
    p = pages[0]
    if p.get("missing") or p.get("invalid") or "disambiguation" in (p.get("pageprops") or {}):
        return None
    text = str(p.get("extract") or "")
    return text if text.strip() else None


def wiki_fetch(entity: str):
    """('page', 본문) 또는 ('similar', [제목들]) 을 반환. 네트워크 실패 시 WikiError.

    논문 환경과 같은 규칙: 문서가 있으면 그 문서, 없으면 비슷한 제목 5개.
    동음이의 문서는 논문 코드처럼 '찾지 못함 + 비슷한 제목'으로 처리한다 (자동으로 골라주지 않음).
    """
    key = entity.strip()
    if key in _PAGE_CACHE:
        return _PAGE_CACHE[key]
    result = None
    if key and not re.search(r"[\[\]{}|#<>]", key):
        text = _load_page(key)
        if text:
            result = ("page", text)
    if result is None:
        titles = _search_titles(key) if key else []
        # 위키백과 검색창의 '바로 이동' 동작: 대소문자만 다른 제목이 있으면 그 문서를 연다
        if titles and titles[0].lower() == key.lower() and titles[0] != key:
            text = _load_page(titles[0])
            if text:
                result = ("page", text)
        if result is None:
            result = ("similar", titles[:5])
    _PAGE_CACHE[key] = result
    return result


def _sentences(page: str) -> list:
    paragraphs = [p.strip() for p in page.split("\n") if p.strip()]
    paragraphs = [p for p in paragraphs if not p.startswith("=")]     # '== History ==' 같은 섹션 제목 제거
    paragraphs = [p for p in paragraphs if len(p.split(" ")) > 2]     # 두 단어 이하 줄 제거 (논문 코드와 동일)
    out = []
    for p in paragraphs:
        out += p.split(". ")
    return [s.strip() if s.strip().endswith(".") else s.strip() + "." for s in out if s.strip()]


class WikiEnv:
    """search[entity] / lookup[keyword] / finish[answer] 세 가지 행동만 받는 환경."""

    def __init__(self, fetch=None):
        self.fetch = fetch or wiki_fetch
        self.page = None
        self.lookup_keyword = None
        self.lookup_list = None
        self.lookup_cnt = None
        self.answer = None
        self.env_errors = 0

    def _search(self, entity: str) -> str:
        try:
            kind, payload = self.fetch(entity)
        except Exception:
            self.env_errors += 1
            return "Could not find that page, please try again."      # Reflexion 공식 코드의 문구
        if kind == "page":
            self.page = payload
            self.lookup_keyword = self.lookup_list = self.lookup_cnt = None
            return " ".join(_sentences(self.page)[:5])                # 첫 5문장
        return f"Could not find {entity}. Similar: {list(payload)[:5]}."

    def _lookup(self, keyword: str) -> str:
        if self.lookup_keyword != keyword:
            self.lookup_keyword = keyword
            sents = _sentences(self.page) if self.page else []
            self.lookup_list = [s for s in sents if keyword.lower() in s.lower()]
            self.lookup_cnt = 0
        if self.lookup_cnt >= len(self.lookup_list):
            return "No more results."
        obs = f"(Result {self.lookup_cnt + 1} / {len(self.lookup_list)}) " + self.lookup_list[self.lookup_cnt]
        self.lookup_cnt += 1
        return obs

    def step(self, action: str):
        """(observation, done) 을 반환. 형식이 틀린 행동은 논문 코드처럼 'Invalid action' 관찰로 돌려준다."""
        a = (action or "").strip()
        if a:
            a = a[0].lower() + a[1:]
        if a.startswith("search[") and a.endswith("]"):
            return self._search(a[len("search["):-1]), False
        if a.startswith("lookup[") and a.endswith("]"):
            return self._lookup(a[len("lookup["):-1]), False
        if a.startswith("finish[") and a.endswith("]"):
            self.answer = a[len("finish["):-1]
            return "Episode finished.", True
        if a.startswith("think[") and a.endswith("]"):
            return "Nice thought.", False
        return f"Invalid action: {a}", False


# ---------------------------------------------------------------------------
# 4. HotpotQA 기준선: Standard / CoT (ReAct 논문 Table 1 의 비교군)
# ---------------------------------------------------------------------------
def _result(method, question, gold, answer="", status="ok", **extra) -> dict:
    out = {"method": method, "question": question, "gold": gold, "answer": answer,
           "correct": bool(gold is not None and status == "ok" and EM(answer, gold)), "status": status}
    out.update(extra)
    return out


def run_standard(question: str, gold=None, llm=ollama_llm) -> dict:
    """Standard 프롬프팅: 질문 → 답 (추론 없음)."""
    try:
        prompt = D.HOTPOT_STANDARD_EXAMPLES + f"Question: {question}\nAnswer:"
        raw = llm(prompt, stop=["\n"], max_tokens=60)
        return _result("standard", question, gold, _strip_label(raw, "Answer:"), raw=raw)
    except LLMError as exc:
        return _result("standard", question, gold, status="llm_error", error=str(exc))
    except Exception as exc:
        return _result("standard", question, gold, status="error", error=f"{type(exc).__name__}: {exc}")


def run_cot(question: str, gold=None, llm=ollama_llm) -> dict:
    """CoT 프롬프팅: 질문 → Thought(단계별 추론) → Answer. 외부 도구 없음."""
    try:
        prompt = D.HOTPOT_COT_EXAMPLES + f"Question: {question}\nThought:"
        raw = llm(prompt, stop=["\nQuestion:"], max_tokens=200)
        text = _strip_label(raw, "Thought:")
        m = re.search(r"^\s*Answer:\s*(.*)$", text, re.M)
        answer = m.group(1).strip() if m else ""
        thought = text.split("\nAnswer:")[0].strip()
        return _result("cot", question, gold, answer, thought=thought, raw=raw,
                       note="" if m else "모델이 'Answer:' 줄을 쓰지 않음 → 오답 처리")
    except LLMError as exc:
        return _result("cot", question, gold, status="llm_error", error=str(exc))
    except Exception as exc:
        return _result("cot", question, gold, status="error", error=f"{type(exc).__name__}: {exc}")


# ---------------------------------------------------------------------------
# 5. ReAct / Act 루프 (ReAct 공식 hotpotqa.ipynb 의 webthink 를 그대로 옮김)
# ---------------------------------------------------------------------------
def format_reflections(reflections: list) -> str:
    if not reflections:
        return ""
    return D.REFLECTION_HEADER + "Reflections:\n- " + "\n- ".join(r.strip() for r in reflections)


def _run_episode(question, gold, llm, env, max_steps, reflections, on_step, use_thought) -> dict:
    method = "react" if use_thought else "act"
    env = env or WikiEnv()
    examples = D.REACT_EXAMPLES if use_thought else D.ACT_EXAMPLES
    prompt = D.REACT_INSTRUCTION + examples
    refl = format_reflections(reflections or [])
    if refl:                                   # Reflexion: 예시 뒤, 질문 앞에 반성 메모리를 넣는다
        prompt += "(END OF EXAMPLES)\n\n" + refl + "\n\n"
    prompt += f"Question: {question}\n"

    steps, n_calls, n_badcalls = [], 0, 0
    done, status, halted = False, "ok", ""
    try:
        for i in range(1, max_steps + 1):
            if not _fits(prompt):
                halted = "context_full"        # Reflexion 공식 코드의 is_halted 와 같은 역할
                break
            n_calls += 1
            if use_thought:
                out = llm(prompt + f"Thought {i}:", stop=[f"\nObservation {i}:"], max_tokens=100)
                out = _strip_label(out, f"Thought {i}:")
                parts = out.split(f"\nAction {i}: ")
                if len(parts) == 2 and parts[1].strip():
                    thought, action = parts[0].strip(), parts[1].strip()
                else:                          # 논문 코드의 'ohh...' 분기: 행동만 다시 요청
                    n_badcalls += 1
                    n_calls += 1
                    lines = out.strip().split("\n")
                    thought = lines[0].strip() if lines else ""
                    action = llm(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=["\n"], max_tokens=100)
                    action = _strip_label(action, f"Action {i}:")
            else:
                thought = ""
                action = _strip_label(llm(prompt + f"Action {i}:", stop=["\n"], max_tokens=100), f"Action {i}:")
            action = action.split("\n")[0].strip()
            obs, done = env.step(action)
            obs = obs.replace("\\n", "").replace("\n", " ").strip()
            if use_thought:
                step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
            else:
                step_str = f"Action {i}: {action}\nObservation {i}: {obs}\n"
            prompt += step_str
            step = {"i": i, "thought": thought, "action": action, "observation": obs}
            steps.append(step)
            if on_step:
                try:
                    on_step(step)
                except Exception:
                    pass
            if done:
                break
        if not done and not halted:
            halted = "max_steps"               # 논문 코드: 7스텝 안에 못 끝내면 빈 답(finish[])으로 종료
    except LLMError as exc:
        status, halted = "llm_error", str(exc)
    except Exception as exc:
        status, halted = "error", f"{type(exc).__name__}: {exc}"

    answer = env.answer if env.answer is not None else ""
    return _result(method, question, gold, answer, status=status, steps=steps, n_calls=n_calls,
                   n_badcalls=n_badcalls, env_errors=env.env_errors, halted=halted, finished=bool(done))


def run_react(question, gold=None, llm=ollama_llm, env=None, max_steps=7, reflections=None, on_step=None) -> dict:
    """ReAct: Thought → Action → Observation 을 최대 7번 반복 (논문 HotpotQA 설정)."""
    return _run_episode(question, gold, llm, env, max_steps, reflections, on_step, use_thought=True)


def run_act(question, gold=None, llm=ollama_llm, env=None, max_steps=7, on_step=None) -> dict:
    """Act: ReAct 에서 Thought 만 뺀 비교군 (논문 Table 1 의 Act)."""
    return _run_episode(question, gold, llm, env, max_steps, None, on_step, use_thought=False)


# ---------------------------------------------------------------------------
# 6. Reflexion (Actor = ReAct, Evaluator = EM, Self-Reflection = LLM, 메모리 = 최근 반성 3개)
# ---------------------------------------------------------------------------
def scratchpad_of(trial: dict, correct: bool) -> str:
    """한 시도의 궤적을 Reflexion 공식 코드의 scratchpad 형식으로 만든다 (끝에 평가자 판정 포함)."""
    lines = []
    for s in trial.get("steps", []):
        obs = s["observation"]
        if s["action"].strip().lower().startswith("finish["):
            obs = "Answer is CORRECT" if correct else "Answer is INCORRECT"
        lines += [f"Thought {s['i']}: {s['thought']}", f"Action {s['i']}: {s['action']}", f"Observation {s['i']}: {obs}"]
    return "\n" + "\n".join(lines) if lines else ""


def truncate_scratchpad(scratchpad: str, n_tokens: int = 1600) -> str:
    """궤적이 길면 가장 긴 Observation 부터 생략 표시로 바꾼다 (Reflexion 공식 코드와 같은 방식)."""
    lines = scratchpad.split("\n")
    obs_idx = sorted((i for i, l in enumerate(lines) if l.startswith("Observation")), key=lambda i: len(lines[i]))
    while len("\n".join(lines)) // 4 > n_tokens and obs_idx:
        i = obs_idx.pop()
        lines[i] = lines[i].split(":")[0] + ": [truncated wikipedia excerpt]"
    return "\n".join(lines)


def make_reflection(question: str, trial: dict, llm=ollama_llm) -> str:
    """실패한 궤적을 보고 '왜 실패했고 다음엔 어떻게 할지'를 말로 쓴다. 실패하면 빈 문자열."""
    try:
        prompt = D.REFLECT_INSTRUCTION.format(
            examples=D.REFLECT_EXAMPLES, question=question,
            scratchpad=truncate_scratchpad(scratchpad_of(trial, correct=False)))
        if not _fits(prompt):
            return ""
        raw = llm(prompt, stop=["\nPrevious Trial:", "\nPrevious trial:", "\nQuestion:"], max_tokens=250)
        return " ".join(_strip_label(raw, "Reflection:").split())
    except Exception:
        return ""


def run_reflexion(question, gold, llm=ollama_llm, max_trials=3, memory_size=3, max_steps=7,
                  env_factory=None, on_step=None, on_event=None) -> dict:
    """Reflexion: 시도 → 평가(EM) → 실패 시 자기 성찰 → 메모리에 넣고 재시도.

    평가자는 논문 HotpotQA 설정대로 '정답과의 정확 일치'다. 그래서 정답(gold)이 반드시 필요하다.
    모델의 가중치는 바뀌지 않는다. 바뀌는 것은 프롬프트에 들어가는 반성 메모뿐이다.
    """
    def emit(kind, payload):
        if on_event:
            try:
                on_event(kind, payload)
            except Exception:
                pass

    trials, reflections, status = [], [], "ok"
    try:
        for t in range(1, max(1, int(max_trials)) + 1):
            memory = reflections[-int(memory_size):] if memory_size else []
            emit("trial_start", {"trial": t, "memory": list(memory)})
            env = env_factory() if env_factory else None
            trial = run_react(question, gold, llm=llm, env=env, max_steps=max_steps,
                              reflections=memory, on_step=on_step)
            trial["trial"] = t
            trial["memory_used"] = list(memory)
            trials.append(trial)
            emit("trial_end", trial)
            if trial["status"] != "ok":
                status = trial["status"]
                break
            if trial["correct"]:
                break
            if t < max_trials:
                refl = make_reflection(question, trial, llm=llm)
                trial["reflection"] = refl
                emit("reflection", {"trial": t, "reflection": refl})
                if refl:
                    reflections.append(refl)
    except Exception as exc:
        status = "error"
        trials.append({"status": "error", "error": f"{type(exc).__name__}: {exc}", "correct": False, "answer": ""})

    last = trials[-1] if trials else {"answer": "", "correct": False}
    out = _result("reflexion", question, gold, last.get("answer", ""), status=status,
                  trials=trials, reflections=reflections, n_trials=len(trials))
    out["correct_by_trial"] = [bool(t.get("correct")) for t in trials]
    return out


# ---------------------------------------------------------------------------
# 7. CoT 논문 실험: 수학 문제에서 Standard vs CoT (few-shot 예시 8개, 논문 Table 20)
# ---------------------------------------------------------------------------
_NUM = r"-?\$?\s*-?\d[\d,]*(?:\.\d+)?"


def extract_number(text: str):
    """'The answer is 18.' 에서 18 을 뽑는다. 그 문장이 없으면 None (오답 처리)."""
    m = re.findall(r"[Tt]he answer is\s*(" + _NUM + ")", text or "")
    if not m:
        return None
    try:
        return float(re.sub(r"[^\d.\-]", "", m[-1]).rstrip("."))
    except ValueError:
        return None


def _same_number(a, b) -> bool:
    try:
        return a is not None and b is not None and abs(float(a) - float(b)) < 1e-6
    except (TypeError, ValueError):
        return False


def run_math(question: str, gold=None, use_cot=True, llm=ollama_llm) -> dict:
    method = "math_cot" if use_cot else "math_standard"
    try:
        examples = D.MATH_COT_EXAMPLES if use_cot else D.MATH_STANDARD_EXAMPLES
        raw = llm(examples + f"Q: {question}\nA:", stop=["\nQ:", "\n\n"], max_tokens=256 if use_cot else 40)
        text = _strip_label(raw, "A:")
        pred = extract_number(text)
        out = _result(method, question, gold, "" if pred is None else pred, raw=text)
        out["correct"] = _same_number(pred, gold)
        if pred is None:
            out["note"] = "모델이 'The answer is N.' 형식으로 끝내지 않음 → 오답 처리"
        return out
    except LLMError as exc:
        return _result(method, question, gold, status="llm_error", error=str(exc))
    except Exception as exc:
        return _result(method, question, gold, status="error", error=f"{type(exc).__name__}: {exc}")


# ---------------------------------------------------------------------------
# 8. 묶음 실행 + 요약 (중간 저장, 이어 하기 지원)
# ---------------------------------------------------------------------------
def _append_jsonl(path, rec):
    if not path:
        return
    try:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    except Exception:
        pass


def load_jsonl(path) -> list:
    out = []
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    out.append(json.loads(line))
                except ValueError:
                    continue
    except OSError:
        pass
    return out


def run_hotpot_batch(items, methods=("standard", "cot", "act", "react", "reflexion"), llm=ollama_llm,
                     max_trials=3, log_path=None, progress=print) -> list:
    """items: [{'question':..., 'answer':...}, ...]. 한 문제·한 방법이 실패해도 나머지는 계속 돈다."""
    done = {(r.get("question"), r.get("method")) for r in load_jsonl(log_path)} if log_path else set()
    records = load_jsonl(log_path) if log_path else []
    runners = {
        "standard": lambda q, g: run_standard(q, g, llm=llm),
        "cot": lambda q, g: run_cot(q, g, llm=llm),
        "act": lambda q, g: run_act(q, g, llm=llm),
        "react": lambda q, g: run_react(q, g, llm=llm),
        "reflexion": lambda q, g: run_reflexion(q, g, llm=llm, max_trials=max_trials),
    }
    for n, item in enumerate(items, 1):
        q, g = str(item.get("question", "")), item.get("answer")
        for m in methods:
            if m not in runners or (q, m) in done:
                continue
            t0 = time.time()
            try:
                rec = runners[m](q, g)
            except Exception as exc:               # 여기까지 오면 안 되지만, 와도 멈추지 않는다
                rec = _result(m, q, g, status="error", error=f"{type(exc).__name__}: {exc}",
                              trace=traceback.format_exc()[-600:])
            rec["seconds"] = round(time.time() - t0, 1)
            records.append(rec)
            _append_jsonl(log_path, rec)
            if progress:
                mark = "⚠️" if rec["status"] != "ok" else ("✅" if rec["correct"] else "❌")
                progress(f"[{n}/{len(items)}] {m:9s} {mark} 답={str(rec['answer'])[:40]!r} 정답={str(g)[:40]!r} ({rec['seconds']}s)")
    return records


def summarize(records: list) -> list:
    """방법별 정답률 표. status 가 ok 가 아닌 실행(통신 실패 등)은 분모에서 빼고 따로 센다."""
    order = ["math_standard", "math_cot", "standard", "cot", "act", "react", "reflexion"]
    rows = []
    for m in order:
        rs = [r for r in records if r.get("method") == m]
        if not rs:
            continue
        ok = [r for r in rs if r.get("status") == "ok"]
        right = sum(1 for r in ok if r.get("correct"))
        row = {"방법": m, "유효 실행": len(ok), "정답": right,
               "정답률(%)": round(100 * right / len(ok), 1) if ok else None,
               "실행 실패": len(rs) - len(ok)}
        if m in ("act", "react"):
            row["형식 오류 호출"] = sum(r.get("n_badcalls", 0) for r in ok)
            row["미완료(스텝 초과)"] = sum(1 for r in ok if not r.get("finished"))
        if m == "reflexion":
            row["1차 시도 정답"] = sum(1 for r in ok if (r.get("correct_by_trial") or [False])[0])
            row["평균 시도 수"] = round(sum(r.get("n_trials", 0) for r in ok) / len(ok), 2) if ok else None
        rows.append(row)
    return rows


# ---------------------------------------------------------------------------
# 9. 준비 상태 점검 (셀마다 먼저 호출 → 준비가 안 됐으면 예외 대신 안내문)
# ---------------------------------------------------------------------------
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        if r.status_code != 200:
            return False, f"Ollama 서버 응답 이상 (HTTP {r.status_code}). ② 셀을 다시 실행하세요."
        names = [m.get("name", "") for m in (r.json().get("models") or [])]
        want = CFG["model"]
        if not any(n == want or n.startswith(want + ":") or n.split(":")[0] == want for n in names):
            return False, f"모델 '{want}' 이(가) 없습니다. 현재 모델: {names}. ② 셀을 다시 실행하세요."
        return True, f"Ollama 정상, 모델 {want} 준비됨"
    except Exception as exc:
        return False, f"Ollama 서버에 연결할 수 없습니다 ({type(exc).__name__}). ② 셀을 먼저 실행하세요."


def check_wikipedia():
    try:
        _PAGE_CACHE.pop("Colorado orogeny", None)
        kind, payload = wiki_fetch("Colorado orogeny")
        if kind == "page" and "orogeny" in payload.lower():
            return True, "Wikipedia 정상: " + " ".join(_sentences(payload)[:1])[:120]
        return False, f"Wikipedia 응답이 예상과 다릅니다: {kind} {str(payload)[:100]}"
    except Exception as exc:
        return False, f"Wikipedia 에 연결할 수 없습니다 ({exc}). 잠시 뒤 다시 실행하세요."


def ready(need_wiki: bool = True) -> bool:
    ok, msg = check_ollama()
    print(("✅ " if ok else "⛔ ") + msg)
    if ok and need_wiki:
        ok2, msg2 = check_wikipedia()
        print(("✅ " if ok2 else "⛔ ") + msg2)
        ok = ok and ok2
    if not ok:
        print("→ 위 문제를 해결한 뒤 이 셀을 다시 실행하세요. (이 셀은 아무것도 실행하지 않고 끝납니다)")
    return ok


def probe_prompt_mode() -> dict:
    """completion / chat 두 방식으로 같은 한 스텝을 시켜 보고, 형식을 지킨 쪽을 알려준다."""
    saved = CFG["prompt_mode"]
    q = "What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?"
    prompt = D.REACT_INSTRUCTION + D.REACT_EXAMPLES + f"Question: {q}\nThought 1:"
    report = {}
    for mode in ("completion", "chat"):
        CFG["prompt_mode"] = mode
        try:
            out = _strip_label(ollama_llm(prompt, stop=["\nObservation 1:"], max_tokens=100), "Thought 1:")
            parts = out.split("\nAction 1: ")
            good = len(parts) == 2 and bool(re.match(r"^(search|lookup|finish)\[.*\]$", parts[1].strip(), re.I))
            report[mode] = {"형식 준수": good, "출력": out.strip()[:200]}
        except Exception as exc:
            report[mode] = {"형식 준수": False, "출력": f"호출 실패: {exc}"}
    CFG["prompt_mode"] = saved
    return report


# ---------------------------------------------------------------------------
# 10. 자가 점검 (가짜 모델·가짜 위키로 로직만 검사. 네트워크·GPU 불필요)
# ---------------------------------------------------------------------------
class ScriptedLLM:
    """미리 정한 출력을 순서대로 내는 가짜 모델. 대본이 떨어지면 빈 문자열."""

    def __init__(self, outputs):
        self.outputs = list(outputs)
        self.prompts = []

    def __call__(self, prompt, stop=("\n",), max_tokens=100):
        self.prompts.append(prompt)
        if not self.outputs:
            return ""
        out = self.outputs.pop(0)
        if isinstance(out, Exception):
            raise out
        return out


_FAKE_WIKI = {
    "Alpha": "Alpha is a town in Beta County. It was founded in 1900. The mayor is Carol Dee. "
             "Alpha has a river.\nHistory\nThe town grew fast. Carol Dee was born in Gamma City.",
    "Gamma City": "Gamma City is the capital of Delta. It has two million people.",
}


def _fake_fetch(entity):
    if entity == "BOOM":
        raise WikiError("network down")
    if entity in _FAKE_WIKI:
        return ("page", _FAKE_WIKI[entity])
    return ("similar", ["Alpha", "Gamma City"])


def selftest(verbose: bool = True) -> bool:
    checks = []

    def check(name, cond):
        checks.append((name, bool(cond)))

    try:
        # 평가자
        check("EM: 관사·대소문자·구두점 무시", EM("The Boston Celtics.", "boston celtics"))
        check("EM: 다른 답은 오답", not EM("Boston", "Boston Celtics"))
        check("EM: None 안전", not EM(None, "x") and EM("", ""))
        # 환경
        env = WikiEnv(fetch=_fake_fetch)
        obs, done = env.step("Search[Alpha]")
        check("search: 첫 문장 반환", obs.startswith("Alpha is a town") and not done)
        check("search: 섹션 제목 제거", "History" not in " ".join(_sentences(_FAKE_WIKI["Alpha"])))
        obs, _ = env.step("Lookup[Carol Dee]")
        check("lookup: 1번째 결과", obs.startswith("(Result 1 / 2)"))
        obs, _ = env.step("Lookup[Carol Dee]")
        check("lookup: 2번째 결과", obs.startswith("(Result 2 / 2)") and "Gamma City" in obs)
        obs, _ = env.step("Lookup[Carol Dee]")
        check("lookup: 결과 소진", obs == "No more results.")
        obs, _ = env.step("Search[Nowhere]")
        check("search: 없는 문서 → Similar", obs.startswith("Could not find Nowhere. Similar: ['Alpha'"))
        obs, _ = env.step("Search[BOOM]")
        check("search: 통신 실패 → 관찰로 처리", "please try again" in obs and env.env_errors == 1)
        obs, _ = env.step("Google[x]")
        check("잘못된 행동 → Invalid action", obs.startswith("Invalid action"))
        obs, _ = env.step("")
        check("빈 행동 → Invalid action", obs.startswith("Invalid action"))
        check("페이지 없이 lookup", WikiEnv(fetch=_fake_fetch).step("lookup[x]")[0] == "No more results.")
        obs, done = env.step("Finish[Gamma City]")
        check("finish: 종료·답 저장", done and env.answer == "Gamma City")

        # ReAct 루프: 정상 경로
        llm = ScriptedLLM([
            " I need to search Alpha.\nAction 1: Search[Alpha]",
            " The mayor is Carol Dee. I need her birthplace.\nAction 2: Lookup[born]",
            " She was born in Gamma City.\nAction 3: Finish[Gamma City]",
        ])
        r = run_react("Where was the mayor of Alpha born?", "Gamma City", llm=llm, env=WikiEnv(fetch=_fake_fetch))
        check("ReAct: 정답 도달", r["correct"] and r["finished"] and len(r["steps"]) == 3 and r["n_badcalls"] == 0)
        check("ReAct: 프롬프트에 관찰 누적", "Observation 2: (Result 1 / 1)" in llm.prompts[2])
        check("ReAct: 프롬프트가 논문 지시문으로 시작", llm.prompts[0].startswith("Solve a question answering task"))

        # ReAct 루프: 형식 깨짐 → 논문 코드의 재요청 분기
        llm = ScriptedLLM(["Thought 1: I should search.", " Search[Alpha]", " done\nAction 2: Finish[Beta County]"])
        r = run_react("q", "Beta County", llm=llm, env=WikiEnv(fetch=_fake_fetch))
        check("ReAct: 행동 누락 시 재요청", r["correct"] and r["n_badcalls"] == 1 and r["steps"][0]["action"] == "Search[Alpha]")

        # ReAct 루프: 끝까지 못 끝냄 / 빈 출력 / 쓰레기 출력
        r = run_react("q", "x", llm=ScriptedLLM([]), env=WikiEnv(fetch=_fake_fetch))
        check("ReAct: 빈 출력이어도 중단 없이 7스텝 후 종료", r["status"] == "ok" and not r["finished"]
              and r["answer"] == "" and not r["correct"] and len(r["steps"]) == 7 and r["halted"] == "max_steps")
        r = run_react("q", "x", llm=ScriptedLLM(["\nAction 1: \nAction 1: ", "###", "[[[", "Finish[", "]"] * 4),
                      env=WikiEnv(fetch=_fake_fetch))
        check("ReAct: 쓰레기 출력에도 예외 없음", r["status"] == "ok" and not r["correct"])
        r = run_react("q", "x", llm=ScriptedLLM([LLMError("down")]), env=WikiEnv(fetch=_fake_fetch))
        check("ReAct: LLM 실패 → status 로 기록", r["status"] == "llm_error" and not r["correct"])
        r = run_react("q", "x", llm=ScriptedLLM([ValueError("odd")]), env=WikiEnv(fetch=_fake_fetch))
        check("ReAct: 예상 밖 예외 → status 로 기록", r["status"] == "error")
        saved = CFG["num_ctx"]
        CFG["num_ctx"] = 500
        r = run_react("q", "x", llm=ScriptedLLM(["a"]), env=WikiEnv(fetch=_fake_fetch))
        CFG["num_ctx"] = saved
        check("ReAct: 컨텍스트 초과 시 호출 없이 중단", r["halted"] == "context_full" and r["n_calls"] == 0)

        # Act
        r = run_act("q", "Gamma City", llm=ScriptedLLM([" Search[Alpha]", " Finish[Gamma City]"]),
                    env=WikiEnv(fetch=_fake_fetch))
        check("Act: Thought 없이 동작", r["correct"] and r["steps"][0]["thought"] == "")

        # Standard / CoT
        check("Standard: 답 추출", run_standard("q", "Yes", llm=ScriptedLLM([" yes"]))["correct"])
        r = run_cot("q", "Ulster County", llm=ScriptedLLM([" Let's think step by step. X is in Ulster County.\nAnswer: Ulster County\n"]))
        check("CoT: Thought·Answer 분리", r["correct"] and r["thought"].startswith("Let's think"))
        r = run_cot("q", "x", llm=ScriptedLLM([" I am not sure."]))
        check("CoT: Answer 줄 없으면 오답", r["status"] == "ok" and not r["correct"] and r["answer"] == "")
        check("Standard: LLM 실패 → status", run_standard("q", "x", llm=ScriptedLLM([LLMError("x")]))["status"] == "llm_error")

        # Reflexion: 1차 실패 → 성찰 → 2차 성공
        llm = ScriptedLLM([
            " Guess.\nAction 1: Finish[Alpha]",
            " I answered without searching. Next time I should search Alpha and look up where the mayor was born.",
            " Search first.\nAction 1: Search[Alpha]",
            " Look up born.\nAction 2: Lookup[born]",
            " Found.\nAction 3: Finish[Gamma City]",
        ])
        r = run_reflexion("Where was the mayor of Alpha born?", "Gamma City", llm=llm, max_trials=3,
                          env_factory=lambda: WikiEnv(fetch=_fake_fetch))
        check("Reflexion: 2차 시도에 정답", r["correct"] and r["n_trials"] == 2 and r["correct_by_trial"] == [False, True])
        check("Reflexion: 성찰 프롬프트에 'INCORRECT' 판정 포함", "Observation 1: Answer is INCORRECT" in llm.prompts[1])
        check("Reflexion: 2차 프롬프트에 반성 메모 포함",
              "Reflections:\n- I answered without searching." in llm.prompts[2] and "(END OF EXAMPLES)" in llm.prompts[2])
        check("Reflexion: 1차 프롬프트는 순수 ReAct 와 동일", "Reflections:" not in llm.prompts[0])

        # Reflexion: 계속 실패 → 최대 시도에서 종료, 메모리 크기 제한
        outs = []
        for k in range(5):
            outs += [f" Guess.\nAction 1: Finish[wrong{k}]", f" reflection number {k}."]
        llm = ScriptedLLM(outs)
        r = run_reflexion("q", "right", llm=llm, max_trials=5, memory_size=3, env_factory=lambda: WikiEnv(fetch=_fake_fetch))
        check("Reflexion: 최대 시도 후 종료", not r["correct"] and r["n_trials"] == 5 and len(r["reflections"]) == 4)
        check("Reflexion: 메모리는 최근 3개만", r["trials"][4]["memory_used"] ==
              ["reflection number 1.", "reflection number 2.", "reflection number 3."])
        r = run_reflexion("q", "right", llm=ScriptedLLM([" x\nAction 1: Finish[a]", LLMError("x"), " y\nAction 1: Finish[right]"]),
                          max_trials=2, env_factory=lambda: WikiEnv(fetch=_fake_fetch))
        check("Reflexion: 성찰 호출 실패해도 재시도 진행", r["correct"] and r["reflections"] == [])
        r = run_reflexion("q", "right", llm=ScriptedLLM([LLMError("x")]), env_factory=lambda: WikiEnv(fetch=_fake_fetch))
        check("Reflexion: Actor 실패 → status", r["status"] == "llm_error" and r["n_trials"] == 1)

        # 궤적 줄이기
        long_pad = "\nThought 1: a\nAction 1: Search[x]\nObservation 1: " + "w " * 5000 + "\nThought 2: b"
        cut = truncate_scratchpad(long_pad)
        check("궤적 줄이기", "[truncated wikipedia excerpt]" in cut and "Thought 2: b" in cut and len(cut) < 400)
        check("궤적 줄이기: Observation 없어도 안전", isinstance(truncate_scratchpad("x" * 20000), str))

        # 수학 CoT
        check("수 추출: 기본", extract_number("5 + 4 = 9. The answer is 9.") == 9)
        check("수 추출: $·쉼표·소수", extract_number("So the answer is $1,234.50.") == 1234.5)
        check("수 추출: 음수", extract_number("The answer is -3") == -3)
        check("수 추출: 마지막 것 사용", extract_number("The answer is 1. No, the answer is 2.") == 2)
        check("수 추출: 형식 없으면 None", extract_number("It is 9.") is None and extract_number("") is None)
        r = run_math("q", 18, use_cot=True, llm=ScriptedLLM([" 16 - 3 - 4 = 9. 9 * 2 = 18. The answer is 18."]))
        check("수학 CoT: 정답 판정", r["correct"])
        r = run_math("q", 18, use_cot=False, llm=ScriptedLLM([" The answer is 20."]))
        check("수학 Standard: 오답 판정", r["status"] == "ok" and not r["correct"])
        check("수학: 형식 없는 출력", not run_math("q", 18, llm=ScriptedLLM(["eighteen"]))["correct"])

        # MediaWiki 응답 해석 (가짜 응답으로 검사)
        global _wiki_get
        real_get, saved_cache = _wiki_get, dict(_PAGE_CACHE)

        def fake_get(**params):
            if params.get("list") == "search":
                return {"query": {"search": [{"title": "High Plains"}, {"title": "High Plains (United States)"}]}}
            t = params.get("titles")
            if t == "High Plains":
                return {"query": {"pages": [{"title": t, "extract": "x may refer to", "pageprops": {"disambiguation": ""}}]}}
            if t == "Real Page":
                return {"query": {"pages": [{"title": t, "extract": "Real Page is a thing. It exists.\n\n== History ==\nIt began long ago."}]}}
            if t == "Down":
                raise WikiError("HTTP 503")
            return {"query": {"pages": [{"title": t, "missing": True}]}}

        try:
            _wiki_get = fake_get
            _PAGE_CACHE.clear()
            check("위키: 문서 본문", wiki_fetch("Real Page")[0] == "page")
            check("위키: 섹션 제목 줄 제거", WikiEnv().step("search[Real Page]")[0] == "Real Page is a thing. It exists. It began long ago.")
            check("위키: 동음이의 → Similar", wiki_fetch("High Plains") == ("similar", ["High Plains", "High Plains (United States)"]))
            check("위키: 대소문자만 다르면 문서 열기 시도", wiki_fetch("real page")[0] == "similar")
            check("위키: 없는 문서 → Similar", wiki_fetch("Zzz")[0] == "similar")
            check("위키: 특수문자 제목 안전", wiki_fetch("a|b[c]")[0] == "similar" and wiki_fetch("")== ("similar", []))
            e = WikiEnv()
            check("위키: 통신 실패 → 관찰", "please try again" in e.step("search[Down]")[0] and e.env_errors == 1)
        finally:
            _wiki_get = real_get
            _PAGE_CACHE.clear()
            _PAGE_CACHE.update(saved_cache)

        # 묶음 실행·요약
        calls = {"n": 0}

        def flaky(prompt, stop=("\n",), max_tokens=100):
            calls["n"] += 1
            if calls["n"] == 2:
                raise LLMError("once")
            return " yes"

        recs = run_hotpot_batch([{"question": "a?", "answer": "yes"}, {"question": "b?", "answer": "yes"},
                                 {"question": "c?", "answer": "no"}], methods=("standard",), llm=flaky, progress=None)
        rows = summarize(recs)
        check("묶음: 실패 1건은 분모에서 제외", len(recs) == 3 and rows[0]["유효 실행"] == 2
              and rows[0]["정답"] == 1 and rows[0]["실행 실패"] == 1 and rows[0]["정답률(%)"] == 50.0)
        check("요약: 빈 입력 안전", summarize([]) == [])

        # 데이터 무결성
        check("데이터: ReAct 예시 6개", D.REACT_EXAMPLES.count("\nQuestion:") == 6 and D.ACT_EXAMPLES.count("Question:") == 6)
        check("데이터: CoT·Standard 예시 6개", D.HOTPOT_COT_EXAMPLES.count("Question:") == 6
              and D.HOTPOT_STANDARD_EXAMPLES.count("Question:") == 6)
        check("데이터: 수학 예시 8개", D.MATH_COT_EXAMPLES.count("\nA:") == 8 == D.MATH_STANDARD_EXAMPLES.count("\nA:"))
        check("데이터: 수학 예시의 답이 두 프롬프트에서 같음",
              re.findall(r"The answer is (\d+)\.", D.MATH_COT_EXAMPLES) == re.findall(r"The answer is (\d+)\.", D.MATH_STANDARD_EXAMPLES))
        check("데이터: 문제 세트", len(D.HOTPOT_DEV) >= 10 and len(D.GSM8K_TEST) >= 10
              and all(i.get("question") and i.get("answer") not in (None, "") for i in D.HOTPOT_DEV + D.GSM8K_TEST))
        check("데이터: 프롬프트가 컨텍스트에 들어감",
              _fits(D.REACT_INSTRUCTION + D.REACT_EXAMPLES, reserve=3000)
              and _fits(D.REFLECT_INSTRUCTION + D.REFLECT_EXAMPLES, reserve=2500))
    except Exception:
        checks.append(("자가 점검 도중 예외:\n" + traceback.format_exc()[-800:], False))

    bad = [n for n, ok in checks if not ok]
    if verbose:
        if bad:
            print(f"⛔ 자가 점검 실패 {len(bad)}건 / {len(checks)}건")
            for n in bad:
                print("   -", n)
        else:
            print(f"✅ 로직 자가 점검 {len(checks)}건 모두 통과 (가짜 모델·가짜 위키 사용)")
    return not bad

## ④ 점검
1. **로직 자가 점검**: 가짜 모델과 가짜 위키로 파싱·루프·채점을 검사합니다. 모델이나 네트워크가 없어도 통과해야 합니다.
2. **연결 점검**: Ollama와 Wikipedia에 실제로 접속해 봅니다.
3. **프롬프트 방식 선택**: 논문은 프롬프트를 그대로 이어 쓰는 completion 방식을 썼습니다. 같은 한 스텝을 `completion`과 `chat` 두 방식으로 시켜 보고, `Thought → Action` 형식을 지키는 쪽을 씁니다. 둘 다 지키면 논문과 같은 `completion`을 씁니다.

In [ ]:
import importlib, sys
READY = False
try:
    for name in ("paper_data", "paper_core"):
        if name in sys.modules:
            importlib.reload(sys.modules[name])
    import paper_data, paper_core as core
    core.CFG["model"] = globals().get("MODEL_NAME", core.CFG["model"])

    logic_ok = core.selftest()
    READY = logic_ok and core.ready(need_wiki=False)
    wiki_ok, wiki_msg = core.check_wikipedia()
    print(("✅ " if wiki_ok else "⚠️ ") + wiki_msg + ("" if wiki_ok else "\n   (⑤ 수학 실험은 Wikipedia 없이도 됩니다. ⑥~⑧은 접속이 될 때 실행하세요.)"))

    if READY:
        report = core.probe_prompt_mode()
        for mode, info in report.items():
            print(f"\n[{mode}] 형식 준수: {info['형식 준수']}\n   출력: {info['출력']!r}")
        if report["completion"]["형식 준수"]:
            core.CFG["prompt_mode"] = "completion"
        elif report["chat"]["형식 준수"]:
            core.CFG["prompt_mode"] = "chat"
        else:
            core.CFG["prompt_mode"] = "completion"
            print("\n⚠️ 두 방식 모두 첫 스텝에서 형식을 지키지 못했습니다. 실험은 그대로 진행되며,"
                  " 형식 오류는 표의 '형식 오류 호출' 칸에 집계됩니다.")
        print(f"\n→ 사용할 프롬프트 방식: {core.CFG['prompt_mode']}   (직접 바꾸려면 core.CFG['prompt_mode'] = 'chat')")
except Exception as exc:
    print("⛔ 점검 중 문제가 생겼습니다. ②·③ 셀을 먼저 실행했는지 확인하세요.\n  ", type(exc).__name__, exc)

## ⑤ CoT 논문 실험: 수학 문제에서 Standard 대 CoT
같은 8개 예시 문제를 쓰되, **Standard**는 예시에 답만 있고 **CoT**는 예시에 풀이 과정이 있습니다. 차이는 그것 하나입니다.
문제는 GSM8K 테스트 세트의 앞 문제들입니다. 답은 모델 출력의 `The answer is N.`에서 뽑아 정답과 숫자로 비교합니다.

In [ ]:
N_MATH = 8          # 풀어 볼 문제 수 (최대 20)
SHOW_FIRST = 2      # 풀이 전문을 보여 줄 문제 수

math_records = []
try:
    if globals().get("READY") and core.ready(need_wiki=False):
        items = paper_data.GSM8K_TEST[:max(1, min(int(N_MATH), len(paper_data.GSM8K_TEST)))]
        for n, item in enumerate(items, 1):
            std = core.run_math(item["question"], item["answer"], use_cot=False)
            cot = core.run_math(item["question"], item["answer"], use_cot=True)
            math_records += [std, cot]
            mark = lambda r: "⚠️" if r["status"] != "ok" else ("✅" if r["correct"] else "❌")
            fmt = lambda r: f"{r['answer']:g}" if isinstance(r["answer"], float) else "(없음)"
            print(f"[{n}/{len(items)}] 정답 {item['answer']:g} | Standard {mark(std)} {fmt(std):>8} | CoT {mark(cot)} {fmt(cot):>8}")
            if n <= SHOW_FIRST:
                print("   문제:", item["question"])
                print("   CoT 풀이:", str(cot.get("raw", cot.get("error", ""))).strip())
                print()
        print()
        for row in core.summarize(math_records):
            print(row)
    else:
        print("④ 셀이 통과한 뒤 다시 실행하세요.")
except Exception as exc:
    print("⛔ 예상하지 못한 문제:", type(exc).__name__, exc)

## ⑥ 한 문제를 네 가지 방법으로 (ReAct 논문 Table 1의 구도)
| 방법 | 추론(Thought) | 행동(Action) |
|---|---|---|
| Standard | 없음 | 없음 |
| CoT | 있음 | 없음 (모델 내부 지식만) |
| Act | 없음 | 있음 |
| ReAct | 있음 | 있음 |

문제는 ReAct 논문 코드와 같은 순서의 HotpotQA dev 문제입니다. `Q_INDEX`를 바꾸면 다른 문제를 봅니다. 직접 만든 질문을 쓰려면 `CUSTOM`에 질문과 정답을 넣으세요 (영어 위키백과로 답할 수 있는 질문이어야 합니다).

In [ ]:
Q_INDEX = 1                      # 0 ~ 29
CUSTOM = None                    # 예: {"question": "Who is older, Danny Green or James Worthy?", "answer": "James Worthy"}

single_records = []


def show_step(s):
    if s["thought"]:
        print(f"  Thought {s['i']}: {s['thought']}")
    print(f"  Action {s['i']}: {s['action']}")
    print(f"  Observation {s['i']}: {s['observation'][:300]}{'…' if len(s['observation']) > 300 else ''}\n")


def verdict(r):
    if r["status"] != "ok":
        return f"⚠️ 실행 실패 ({r['status']}: {r.get('error') or r.get('halted')})"
    return ("✅ 정답" if r["correct"] else "❌ 오답") + f" | 모델 답: {r['answer']!r} | 정답: {r['gold']!r}"


try:
    if globals().get("READY") and core.ready(need_wiki=True):
        item = CUSTOM or paper_data.HOTPOT_DEV[max(0, min(int(Q_INDEX), len(paper_data.HOTPOT_DEV) - 1))]
        q, gold = item["question"], item["answer"]
        print("질문:", q, "\n정답:", gold, "\n")

        print("━━ Standard ━━")
        r = core.run_standard(q, gold); single_records.append(r); print(" ", verdict(r), "\n")

        print("━━ CoT ━━")
        r = core.run_cot(q, gold); single_records.append(r)
        print("  Thought:", r.get("thought", "")); print(" ", verdict(r), "\n")

        print("━━ Act ━━")
        r = core.run_act(q, gold, on_step=show_step); single_records.append(r); print(" ", verdict(r), "\n")

        print("━━ ReAct ━━")
        r = core.run_react(q, gold, on_step=show_step); single_records.append(r)
        print(" ", verdict(r), f"| 형식 오류 호출 {r.get('n_badcalls', 0)}회", "\n")
    else:
        print("④ 셀이 통과한 뒤 다시 실행하세요.")
except Exception as exc:
    print("⛔ 예상하지 못한 문제:", type(exc).__name__, exc)

## ⑦ Reflexion: 틀리면 말로 반성하고 다시 시도
- **Actor**: ⑥의 ReAct와 같습니다. 1차 시도는 순수 ReAct입니다.
- **Evaluator**: 정답과의 정확 일치(EM). 논문 HotpotQA 설정과 같습니다. 모델이 스스로 채점하지 않습니다.
- **Self-Reflection**: 실패한 궤적을 보고 원인과 다음 계획을 문장으로 씁니다.
- **메모리**: 최근 반성 3개를 다음 시도의 프롬프트에 넣습니다. 모델 가중치는 바뀌지 않습니다.

1차 시도에서 맞히면 반성이 일어나지 않습니다. `FIND_FAILURE = True`이면 1차에서 틀리는 문제가 나올 때까지 문제를 넘겨 가며 찾습니다.

In [ ]:
MAX_TRIALS = 3          # 한 문제당 최대 시도 수
FIND_FAILURE = True     # True: 1차 시도에서 틀리는 문제를 찾아서 보여 줌 / False: START_INDEX 문제만
START_INDEX = 0
MAX_SEARCH = 8          # FIND_FAILURE 일 때 살펴볼 최대 문제 수

reflexion_demo = None


def on_event(kind, p):
    if kind == "trial_start":
        print(f"\n━━ 시도 {p['trial']} ━━" + (f"  (메모리의 반성 {len(p['memory'])}개 사용)" if p["memory"] else "  (반성 없음 = 순수 ReAct)"))
    elif kind == "trial_end":
        print(" ", verdict(p))
    elif kind == "reflection":
        print("\n  💭 자기 성찰:", p["reflection"] or "(생성 실패 → 반성 없이 재시도)")


try:
    if globals().get("READY") and core.ready(need_wiki=True):
        pool = paper_data.HOTPOT_DEV[int(START_INDEX):]
        pool = pool[:int(MAX_SEARCH)] if FIND_FAILURE else pool[:1]
        for k, item in enumerate(pool):
            print(f"\n{'=' * 70}\n질문: {item['question']}\n정답: {item['answer']}")
            r = core.run_reflexion(item["question"], item["answer"], max_trials=MAX_TRIALS,
                                   on_step=show_step, on_event=on_event)
            reflexion_demo = r
            if r["status"] != "ok":
                print("⚠️ 실행 실패:", r["status"]); continue
            if r["n_trials"] > 1 or not FIND_FAILURE:
                break
            print("→ 1차 시도에서 맞혔습니다. 반성이 필요 없는 문제이므로 다음 문제로 넘어갑니다.")
        if reflexion_demo:
            r = reflexion_demo
            print(f"\n{'=' * 70}\n시도별 결과: {['정답' if c else '오답' for c in r['correct_by_trial']]}")
            print("최종:", verdict(r))
            if r["n_trials"] > 1 and not r["correct"]:
                print("※ 반성을 거쳐도 못 맞힌 경우입니다. 논문도 Reflexion이 국소 최솟값에 갇힐 수 있다고 한계로 적었습니다.")
    else:
        print("④ 셀이 통과한 뒤 다시 실행하세요.")
except Exception as exc:
    print("⛔ 예상하지 못한 문제:", type(exc).__name__, exc)

## ⑧ 묶음 실험: 같은 문제들에 다섯 방법을 모두 적용
결과는 한 건이 끝날 때마다 `results/hotpot_runs.jsonl`에 저장됩니다. 중간에 끊겨도 이 셀을 다시 실행하면 **끝난 것은 건너뛰고 이어서** 합니다. 처음부터 다시 하려면 `RESET = True`로 두세요.

걸리는 시간은 GPU 종류에 따라 크게 다릅니다 (아직 실측 전). 문제 수가 적으면 정답률의 오차가 큽니다. 5문제에서 한 문제 차이는 20%p이므로, 발표에서는 "N문제 중 몇 문제"로 말하는 편이 정확합니다.

In [ ]:
N_QUESTIONS = 5                                   # 문제 수 (최대 30)
METHODS = ("standard", "cot", "act", "react", "reflexion")
MAX_TRIALS = 3
RESET = False
LOG_PATH = "results/hotpot_runs.jsonl"

batch_records = []
try:
    if globals().get("READY") and core.ready(need_wiki=True):
        import os
        if RESET and os.path.exists(LOG_PATH):
            os.remove(LOG_PATH)
        items = paper_data.HOTPOT_DEV[:max(1, min(int(N_QUESTIONS), len(paper_data.HOTPOT_DEV)))]
        batch_records = core.run_hotpot_batch(items, methods=METHODS, max_trials=MAX_TRIALS, log_path=LOG_PATH)
        wanted = {i["question"] for i in items}
        batch_records = [r for r in batch_records if r.get("question") in wanted and r.get("method") in METHODS]
        rows = core.summarize(batch_records)
        print(f"\n모델 {core.CFG['model']} · 프롬프트 방식 {core.CFG['prompt_mode']} · 문제 {len(items)}개\n")
        try:
            import pandas as pd
            from IPython.display import display
            display(pd.DataFrame(rows).fillna(""))
        except Exception:
            for row in rows:
                print(row)
        print("\n통신 통계:", core.STATS)
    else:
        print("④ 셀이 통과한 뒤 다시 실행하세요.")
except Exception as exc:
    print("⛔ 예상하지 못한 문제:", type(exc).__name__, exc)

### 결과를 읽는 법
- **정답률(%)** 은 "유효 실행" 중 정답 비율입니다. 통신 실패 등으로 실행 자체가 안 된 건은 "실행 실패"로 따로 셉니다.
- **형식 오류 호출**: 모델이 `Thought → Action` 형식을 못 지켜 행동을 다시 요청한 횟수입니다 (ReAct 공식 코드의 `n_badcalls`와 같습니다).
- **미완료(스텝 초과)**: 7스텝 안에 `finish`를 못 한 문제 수입니다. 논문 코드처럼 빈 답으로 처리되어 오답입니다.
- **Reflexion의 "1차 시도 정답"** 은 반성 없이 맞힌 수입니다. "정답"에서 이 값을 뺀 것이 반성으로 얻은 개선입니다.
- EM은 엄격합니다. `"Seth MacFarlane"`이 정답일 때 `"Seth MacFarlane and Mike Barker"`는 오답입니다. 논문도 같은 기준입니다.

### 논문 수치와 다르게 나오는 것이 정상인 이유
논문의 ReAct는 PaLM-540B, Reflexion은 GPT 계열을 썼습니다. 8B 모델은 예시 형식을 따라 하는 능력과 다단계 추론 능력이 모두 약합니다. ReAct 논문의 오류 분석에서도 ReAct 실패의 가장 큰 원인은 추론 오류(47%)였고, 이는 모델이 작을수록 커집니다.

## ⑨ 결과 내려받기
발표 자료에 넣을 수 있도록 실행 기록과 요약을 파일로 묶습니다.

In [ ]:
try:
    import json, os, zipfile
    os.makedirs("results", exist_ok=True)
    everything = {
        "model": core.CFG["model"], "prompt_mode": core.CFG["prompt_mode"],
        "math": globals().get("math_records", []),
        "single_question": globals().get("single_records", []),
        "reflexion_demo": globals().get("reflexion_demo"),
        "batch": globals().get("batch_records", []),
    }
    everything["summary"] = core.summarize(everything["math"] + everything["batch"])
    with open("results/all_results.json", "w", encoding="utf-8") as f:
        json.dump(everything, f, ensure_ascii=False, indent=1)
    with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for name in os.listdir("results"):
            z.write(os.path.join("results", name))
    print("✅ results.zip 생성 (왼쪽 파일 탭에서도 받을 수 있습니다)")
    for row in everything["summary"]:
        print(row)
    try:
        from google.colab import files
        files.download("results.zip")
    except Exception:
        pass
except Exception as exc:
    print("⛔ 저장 중 문제:", type(exc).__name__, exc, "\n②~④ 셀을 먼저 실행했는지 확인하세요.")